# Libraries management

In [1]:
import cx_Oracle
import oracledb as odb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import win32com.client as win32
import gc as gc
from time import sleep
import sys as sys
import os as os
from datetime import datetime as datetime
import pythoncom as pythoncom
# from minio import Minio
# import smtplib

## Check python version
import sys
import platform

In [2]:
# print(sys.version)
# print(platform.python_version())

# Config management

In [3]:
instant_client_path = r"D:\oracle\instantclient_19_29"
cx_Oracle.init_oracle_client(lib_dir=instant_client_path)
print("Initialization successful.")

Initialization successful.


# Declaration of variables

In [4]:
ip = 'ora-dwhdb-pri.centralretail.com.vn'
port = '1521'
service_name = 'DWHDB'
username = 'omni_digimgr'
password = 'Buzz@191996'
# password = 'TtJ6zr3J5o'
sql_script = """
SELECT
	*
FROM
	crv_data.loutruong_supplier_perf_di
WHERE
	1 = 1
	AND sale_date >= ADD_MONTHS(TRUNC(SYSDATE - 1, 'YYYY'), -12)
	AND supplier_code IN (
		SELECT
			supplier_code
		FROM
			omni_digimgr.loutruong_dim_supplier
	)
ORDER BY
	sale_date ASC,
	supplier_code ASC,
	dimension_group ASC,
	dimension ASC
"""
sheet_path = r"D:\OneDrive - Central Group\Stella's files - 1. HAND OVER\3. REPORT DAILY\09_supplier_tracker\Supplier_Performance_Tracker.xlsx"
sheet_name = 'perf_raw_di'

# 3. Connection controller

In [5]:
dsn_tns = cx_Oracle.makedsn(ip, port, service_name=service_name)
conn = cx_Oracle.connect(
    user=username, password=password, dsn=dsn_tns).cursor()

In [6]:
query_output = conn.execute(sql_script)
title = [i[0] for i in conn.description]
df = pd.DataFrame(data=query_output, columns=title)

In [7]:
# print(df.head(10).to_string())

# print(df.dtypes)

# Main logic: Open > Delete > Write > Close

In [8]:
xl = win32.DispatchEx("Excel.Application")
xl.Application.Visible = False  # Open in background. should be True for debugging
xl.Application.DisplayAlerts = False  # To avoid any pop ups
wb = xl.Workbooks.Open(sheet_path)
sleep(3) ### To wait application load, increase if needed, 3 is number of seconds to wait.
wb.RefreshAll() ## Refresh all in excel
xl.CalculateUntilAsyncQueriesDone()
sleep(2) ## To ensure all things is done
print('Done Refresh')

Done Refresh


In [9]:
# --- Grab the specific worksheet object ---
try:
    ws = wb.Sheets(sheet_name)
except Exception as e:
    print(f"Error: Could not find sheet '{sheet_name}'. Please check the name.")
    raise e

In [10]:
# --- Clear existing data ---
def column_to_letter(col_num):
    string = ""
    while col_num > 0:
        col_num, remainder = divmod(col_num - 1, 26) 
        string = chr(65 + remainder) + string
    return string

last_col = ws.UsedRange.Column + ws.UsedRange.Columns.Count - 1
last_row = ws.UsedRange.Row + ws.UsedRange.Rows.Count - 1

if last_row < 2:
    print(f"Sheet '{sheet_name}' is empty or only contains a header row. No data was cleared.")
    
else:
    start_cell = ws.Cells(2, 1) 
    end_cell = ws.Cells(last_row, last_col) 
    clear_range = ws.Range(start_cell, end_cell)
    clear_range.ClearContents()
    last_col_letter = column_to_letter(last_col)
    print(f"Cleared data from Row 2 to {last_row} (Columns A to {last_col_letter}) in sheet '{sheet_name}'.")

Cleared data from Row 2 to 485112 (Columns A to L) in sheet 'perf_raw_di'.


In [11]:
# --- Paste new data from dataframe ---
df_clean = df.copy()
df_clean['SALE_DATE'] = df_clean['SALE_DATE'].dt.strftime('%Y-%m-%d').fillna('')

data_to_paste = df_clean.values.tolist()
num_rows = len(df_clean)
num_cols = len(df_clean.columns)

if num_rows > 0:
    start_row = 2
    start_col = 1
    end_row = start_row + num_rows - 1
    end_col = num_cols 
    target_range = ws.Range(ws.Cells(start_row, start_col), ws.Cells(end_row, end_col))
    target_range.Value = data_to_paste
    print(f"Successfully pasted {num_rows} rows into sheet '{sheet_name}', starting at A2.")
    print(f"SCRIPT_RESULT: SUCCESS; ROWS_PROCESSED: {num_rows}")
else:
    print("DataFrame is empty. Nothing to paste.")
    print(f"SCRIPT_RESULT: SUCCESS; ROWS_PROCESSED: 0")

Successfully pasted 323363 rows into sheet 'perf_raw_di', starting at A2.
SCRIPT_RESULT: SUCCESS; ROWS_PROCESSED: 323363


In [12]:
# wb.Save()
# sleep(2)
# xl.Application.Quit()
# xl = None
# del xl
# pythoncom.CoUninitialize()
# gc.collect()

In [13]:
# 1. Close the workbook specifically before quitting
if 'wb' in locals():
    wb.Close(SaveChanges=True) 

# 2. Quit the application
xl.Application.Quit()

# 3. Explicitly delete references to COM objects
# This is the crucial step to release the interface
del wb
del xl

# 4. Give COM a moment to release resources
sleep(2)

# 5. Clean up the COM library
pythoncom.CoUninitialize()

# 6. Force garbage collection
gc.collect()

10